# Fig. 2B: pseudotime APA volcano (rho_W), half-width, lineages stacked

Half-width panel; excitatory (top) over inhibitory (bottom) so it pairs side-by-side with the
treatment volcano (`fig3a_treatment_volcano`). x = Spearman rho_W (3'UTR-length index W vs
pseudotime, |rho_W|>0.2 gate), y = -log10(LRT padj). ASD-risk genes starred. The Delta-psi
version is S3b in `supp_3_developmental_landscape`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

PANEL = "fig2c_pseudotime_volcano"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(84)  # 85 x 84 mm (half-grid, 2 stacked)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)
KB = f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling"
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
PADJ_THR = 0.05
RHO_THR = 0.2
ASD_STAR = "#e08214"

In [ ]:
comp = {
    lin: pd.read_csv(f"{KB}/composite_ranking/composite_ranking_optimized2_{lin}.csv")
    for lin, _ in LINS
}
asd = pd.read_csv(f"{KB}/asd_results/asd_significant_in_knoblich.csv")
for lin, _ in LINS:
    d = comp[lin]
    d["nl"] = -np.log10(d["padj"].clip(lower=1e-300))
    cap = d.loc[np.isfinite(d["nl"]), "nl"].quantile(0.999)
    d["nl"] = d["nl"].clip(upper=cap * 1.05)
    d["sig"] = np.where(
        (d["padj"] < PADJ_THR) & (d["rho_W"] < -RHO_THR),
        "short",
        np.where((d["padj"] < PADJ_THR) & (d["rho_W"] > RHO_THR), "long", "NS"),
    )
    print(f"{lin}: short={int((d.sig == 'short').sum())} long={int((d.sig == 'long').sum())}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=FIGSIZE)
for ax, (lin, linlabel) in zip(axes, LINS):
    d = comp[lin]
    cs = d["composite_score_abs"].values
    size = 1 + 10 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
    for cls, col, z in [
        ("NS", S.COL_NS, 1),
        ("short", S.COL_SHORT, 2),
        ("long", S.COL_LONG, 2),
    ]:
        m = (d["sig"] == cls).values
        ax.scatter(
            d.loc[m, "rho_W"],
            d.loc[m, "nl"],
            c=col,
            s=size[m],
            alpha=0.35 if cls == "NS" else 0.7,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    ax.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.5, zorder=0)
    for xv in (-RHO_THR, RHO_THR):
        ax.axvline(xv, color="gray", ls="--", lw=0.5, zorder=0)
    ax.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.20)
    a = asd[asd["lineage"] == lin]
    if len(a):
        am = d["gene_name"].isin(a["gene_name"]).values
        ax.scatter(
            d.loc[am, "rho_W"],
            d.loc[am, "nl"],
            s=14,
            marker="*",
            facecolors="none",
            edgecolors=ASD_STAR,
            linewidths=0.5,
            zorder=5,
        )
    top = d[d["sig"] != "NS"].nlargest(4, "composite_score_abs")
    texts = [
        ax.text(
            r["rho_W"],
            r["nl"],
            r["gene_name"],
            fontsize=S.FONT_ANNOT - 1,
            fontstyle="italic",
            zorder=10,
        )
        for _, r in top.iterrows()
    ]
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    ns = int((d["sig"] == "short").sum())
    nl_ = int((d["sig"] == "long").sum())
    ax.text(
        0.02,
        0.96,
        f"{ns}",
        transform=ax.transAxes,
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="left",
    )
    ax.text(
        0.98,
        0.96,
        f"{nl_}",
        transform=ax.transAxes,
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="right",
    )
    ax.set_title(
        linlabel,
        fontsize=S.FONT_TICK,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=1,
    )
    ax.set_ylabel(r"$-\log_{10}p_{\mathrm{adj}}$", fontsize=S.FONT_TICK)
    ax.tick_params(labelsize=S.FONT_TICK - 1)
axes[1].set_xlabel(r"Spearman $\rho_W$  (3$^\prime$UTR length vs pseudotime)", fontsize=S.FONT_TICK)
leg = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_SHORT,
        markersize=3.5,
        label="shortening",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_LONG,
        markersize=3.5,
        label="lengthening",
    ),
    Line2D(
        [0],
        [0],
        marker="*",
        color="w",
        markerfacecolor="none",
        markeredgecolor=ASD_STAR,
        markersize=5,
        label="ASD-risk",
    ),
]
axes[0].legend(
    handles=leg,
    fontsize=S.FONT_LEGEND - 1,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.10),
    ncol=3,
    handletextpad=0.15,
    columnspacing=0.6,
)
fig.subplots_adjust(left=0.15, right=0.97, top=0.88, bottom=0.10, hspace=0.42)
S.save_and_show(fig, PANEL, figure="Fig2")

In [ ]:
# Permutation-calibrated p-value + permutation FDR per lineage (as in Fig 1d).
# Companion "pseudo p-value" (raw chi-sq magnitude, permutation-FDR-coloured) is the primary
# alternative; the perm-calibrated deflation is a sensitivity (deflation is mild - same
# finding as mouse Fig 1d: the permutation null confirms the chi-sq LRT calibration).
for lin, _ in LINS:
    d = comp[lin].copy()
    permf = (
        f"{KB}/dirmult_results/{lin}/permutation_batch_number/"
        f"dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_{lin}_cov-batch_number.csv"
    )
    pm = pd.read_csv(permf)[["utr_name", "p_value_perm", "n_perm_hits"]]
    d = d.merge(pm, left_on="gene_id", right_on="utr_name", how="left")
    cal = S.permutation_calibrated_p(
        d["p_value_chisq"].values,
        d["p_value_perm"].values,
        at_floor=(d["n_perm_hits"].fillna(0).values == 0),
    )
    d["neglog_cal"] = cal["neglog_cal"]
    d["extrap"] = cal["extrapolated"]
    d["padj_perm"] = S.benjamini_hochberg(d["p_value_perm"].values)
    d["nl_chisq"] = -np.log10(d["p_value_chisq"].clip(lower=1e-300))
    d["sig_perm"] = np.where(
        (d["padj_perm"] < PADJ_THR) & (d["rho_W"] < -RHO_THR),
        "short",
        np.where((d["padj_perm"] < PADJ_THR) & (d["rho_W"] > RHO_THR), "long", "NS"),
    )
    comp[lin] = d
    print(
        f"{lin}: perm-FDR short={int((d.sig_perm == 'short').sum())} long={int((d.sig_perm == 'long').sum())} "
        f"| chi-sq gate short={int((d.sig == 'short').sum())} long={int((d.sig == 'long').sum())} "
        f"| cal slope={cal['slope']:.2f} maxnl {d['nl_chisq'].max():.0f}->{d['neglog_cal'].max():.0f}"
    )


def render_fig2c(ycol, ylabel, panel, extrap=False, figsize=None, ntop=4, compact=False):
    fig, axes = plt.subplots(2, 1, figsize=figsize if figsize is not None else FIGSIZE)
    for ax, (lin, linlabel) in zip(axes, LINS):
        d = comp[lin]
        y = d[ycol].values
        cs = d["composite_score_abs"].values
        size = 1 + 10 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
        for cls, col, z in [
            ("NS", S.COL_NS, 1),
            ("short", S.COL_SHORT, 2),
            ("long", S.COL_LONG, 2),
        ]:
            m = (d["sig_perm"] == cls).values
            ax.scatter(
                d.loc[m, "rho_W"],
                y[m],
                c=col,
                s=size[m],
                alpha=0.35 if cls == "NS" else 0.7,
                edgecolors="none",
                rasterized=True,
                zorder=z,
            )
        for xv in (-RHO_THR, RHO_THR):
            ax.axvline(xv, color="gray", ls="--", lw=0.5, zorder=0)
        ax.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)
        if extrap:
            yy = y[d["extrap"].values & (d["sig_perm"] != "NS").values]
            if yy.size:
                ax.axhspan(
                    float(np.nanmin(yy)),
                    ax.get_ylim()[1],
                    color="#fff2cc",
                    alpha=0.4,
                    zorder=0,
                )
        ax.set_ylim(top=ax.get_ylim()[1] * 1.20)
        a = asd[asd["lineage"] == lin]
        if len(a):
            am = d["gene_name"].isin(a["gene_name"]).values
            ax.scatter(
                d.loc[am, "rho_W"],
                y[am],
                s=14,
                marker="*",
                facecolors="none",
                edgecolors=ASD_STAR,
                linewidths=0.5,
                zorder=5,
            )
        top = d[d["sig_perm"] != "NS"].nlargest(ntop, "composite_score_abs")
        texts = [
            ax.text(
                r["rho_W"],
                yi,
                r["gene_name"],
                fontsize=S.FONT_ANNOT - 1,
                fontstyle="italic",
                zorder=10,
            )
            for (_, r), yi in zip(top.iterrows(), y[top.index.values])
        ]
        if HAS_AT and texts:
            adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
        ns = int((d.sig_perm == "short").sum())
        nl_ = int((d.sig_perm == "long").sum())
        ax.text(
            0.02,
            0.96,
            f"{ns}",
            transform=ax.transAxes,
            color=S.COL_SHORT,
            fontsize=S.FONT_ANNOT - 0.5,
            fontweight="bold",
            va="top",
            ha="left",
        )
        ax.text(
            0.98,
            0.96,
            f"{nl_}",
            transform=ax.transAxes,
            color=S.COL_LONG,
            fontsize=S.FONT_ANNOT - 0.5,
            fontweight="bold",
            va="top",
            ha="right",
        )
        ax.set_title(
            linlabel,
            fontsize=S.FONT_TICK,
            color=S.paper_palette_lineage[lin],
            fontweight="bold",
            pad=1,
        )
        ax.set_ylabel(ylabel, fontsize=S.FONT_TICK)
        ax.tick_params(labelsize=S.FONT_TICK - 1)
    axes[1].set_xlabel(
        r"Spearman $\rho_W$  (3$^\prime$UTR length vs pseudotime)", fontsize=S.FONT_TICK
    )
    leg = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_SHORT,
            markersize=3.5,
            label="shortening",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_LONG,
            markersize=3.5,
            label="lengthening",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            markerfacecolor="none",
            markeredgecolor=ASD_STAR,
            markersize=5,
            label="ASD-risk",
        ),
    ]
    axes[0].legend(
        handles=leg,
        fontsize=S.FONT_LEGEND - (2 if compact else 1),
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.16 if compact else 1.10),
        ncol=3,
        handletextpad=0.15,
        columnspacing=0.6,
    )
    if compact:
        fig.subplots_adjust(left=0.15, right=0.97, top=0.82, bottom=0.15, hspace=0.62)
    else:
        fig.subplots_adjust(left=0.15, right=0.97, top=0.88, bottom=0.10, hspace=0.42)
    S.save_and_show(fig, panel, figure="Fig2")

### Alternative panel: permutation-based p-value

As in Fig. 1D. The primary companion below shows the raw chi-sq magnitude labelled as an uncalibrated 'pseudo p-value', coloured by permutation FDR (the assumption-light null). As in mouse, the perm-calibrated deflation (last cell) is mild, i.e. the permutation null confirms the chi-sq calibration, so the companion is the primary alternative.


In [ ]:
# Primary alternative (companion): raw chi-sq magnitude relabeled uncalibrated; colour =
# permutation FDR (+ the |rho_W| > 0.2 effect-size gate).
render_fig2c(
    "nl_chisq",
    r"$-\log_{10}p$  (nominal, uncalibrated)",
    "fig2c_pseudotime_volcano__pseudo_chisq",
)

### Variants: inhibitory-only pseudo-chisq volcano (half-width; quarter- & eighth-height)

For an inhibitory-only layout of Fig. 2: the INH pseudotime volcano alone (companion "pseudo
p-value" = raw chi-sq magnitude, permutation-FDR coloured, |rho_W|>0.2 gate), to pair
with the inhibitory UMAP. Two heights for comparison. Saved as
`fig2c_pseudotime_volcano__pseudo_chisq_inh_quarter` and `..._inh_eighth`.

In [ ]:
# --- Inhibitory-only pseudo-chisq volcano variants (half-width; quarter- and eighth-height) ---
# INH pseudotime volcano alone (raw chi-sq magnitude, permutation-FDR coloured, |rho_W|>0.2
# gate) to pair with the inhibitory UMAP. Two heights for comparison. Reuses `comp` (with
# nl_chisq/sig_perm computed above), `asd`, RHO_THR, ASD_STAR, HAS_AT/adjust_text.
def _volcano_inh(figsize, save_name, *, compact):
    d = comp["inhibitory"]
    y = d["nl_chisq"].values
    cs = d["composite_score_abs"].values
    size = 1 + 10 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
    fig, ax = plt.subplots(figsize=figsize)
    for cls, col, z in [("NS", S.COL_NS, 1), ("short", S.COL_SHORT, 2), ("long", S.COL_LONG, 2)]:
        m = (d["sig_perm"] == cls).values
        ax.scatter(
            d.loc[m, "rho_W"],
            y[m],
            c=col,
            s=size[m],
            alpha=0.35 if cls == "NS" else 0.7,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    for xv in (-RHO_THR, RHO_THR):
        ax.axvline(xv, color="gray", ls="--", lw=0.5, zorder=0)
    ax.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)
    a = asd[asd["lineage"] == "inhibitory"]
    if len(a):
        am = d["gene_name"].isin(a["gene_name"]).values
        ax.scatter(
            d.loc[am, "rho_W"],
            y[am],
            s=14,
            marker="*",
            facecolors="none",
            edgecolors=ASD_STAR,
            linewidths=0.5,
            zorder=5,
        )
    ntop = 3 if compact else 4
    top = d[d["sig_perm"] != "NS"].nlargest(ntop, "composite_score_abs")
    texts = [
        ax.text(
            r["rho_W"], yi, r["gene_name"], fontsize=S.FONT_ANNOT - 1, fontstyle="italic", zorder=10
        )
        for (_, r), yi in zip(top.iterrows(), y[top.index.values])
    ]
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    ax.text(
        0.02,
        0.96,
        f"{int((d.sig_perm == 'short').sum())}",
        transform=ax.transAxes,
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="left",
    )
    ax.text(
        0.98,
        0.96,
        f"{int((d.sig_perm == 'long').sum())}",
        transform=ax.transAxes,
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="right",
    )
    ax.set_ylabel(r"$-\log_{10}p$ (nominal)", fontsize=S.FONT_TICK)
    ax.set_xlabel(r"Spearman $\rho_W$  (3$^\prime$UTR length vs pseudotime)", fontsize=S.FONT_TICK)
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    leg = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_SHORT,
            markersize=3.5,
            label="shortening",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_LONG,
            markersize=3.5,
            label="lengthening",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            markerfacecolor="none",
            markeredgecolor=ASD_STAR,
            markersize=5,
            label="ASD-risk",
        ),
    ]
    if compact:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1.5,
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(0.0, 0.86),
            handletextpad=0.1,
            labelspacing=0.15,
            borderpad=0.1,
        )
        fig.subplots_adjust(left=0.14, right=0.975, top=0.965, bottom=0.30)
    else:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1,
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=3,
            handletextpad=0.15,
            columnspacing=0.6,
        )
        ax.set_title(
            "Inhibitory neurons",
            fontsize=S.FONT_TICK,
            color=S.paper_palette_lineage["inhibitory"],
            fontweight="bold",
            pad=14,
        )
        fig.subplots_adjust(left=0.135, right=0.975, top=0.80, bottom=0.155)
    S.save_and_show(fig, save_name, figure="Fig2")


_volcano_inh(
    (S.HALF_W_IN, S.QUARTER_H_IN),
    "fig2c_pseudotime_volcano__pseudo_chisq_inh_quarter",
    compact=False,
)
_volcano_inh(
    (S.HALF_W_IN, S.EIGHTH_H_IN), "fig2c_pseudotime_volcano__pseudo_chisq_inh_eighth", compact=True
)

### Variant: excitatory pseudo-chisq volcano (half-width; quarter- & eighth-height)

The excitatory counterpart of the inhibitory volcano above (raw chi-sq magnitude,
permutation-FDR coloured, |rho_W|>0.2 gate). Saved as
`fig2c_pseudotime_volcano__pseudo_chisq_exc_quarter` and `..._exc_eighth`.

In [ ]:
# --- Excitatory pseudo-chisq volcano variants (half-width; quarter- and eighth-height) ---
# EXC counterpart of the inhibitory volcano. Reuses `comp` (nl_chisq/sig_perm), `asd`,
# RHO_THR, ASD_STAR, HAS_AT/adjust_text, Line2D.
def _volcano_lin(lin, linlabel, figsize, save_name, *, compact):
    d = comp[lin]
    y = d["nl_chisq"].values
    cs = d["composite_score_abs"].values
    size = 1 + 10 * (cs - np.nanmin(cs)) / (np.nanmax(cs) - np.nanmin(cs) + 1e-9)
    fig, ax = plt.subplots(figsize=figsize)
    for cls, col, z in [("NS", S.COL_NS, 1), ("short", S.COL_SHORT, 2), ("long", S.COL_LONG, 2)]:
        m = (d["sig_perm"] == cls).values
        ax.scatter(
            d.loc[m, "rho_W"],
            y[m],
            c=col,
            s=size[m],
            alpha=0.35 if cls == "NS" else 0.7,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    for xv in (-RHO_THR, RHO_THR):
        ax.axvline(xv, color="gray", ls="--", lw=0.5, zorder=0)
    ax.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)
    a = asd[asd["lineage"] == lin]
    if len(a):
        am = d["gene_name"].isin(a["gene_name"]).values
        ax.scatter(
            d.loc[am, "rho_W"],
            y[am],
            s=14,
            marker="*",
            facecolors="none",
            edgecolors=ASD_STAR,
            linewidths=0.5,
            zorder=5,
        )
    ntop = 3 if compact else 4
    top = d[d["sig_perm"] != "NS"].nlargest(ntop, "composite_score_abs")
    texts = [
        ax.text(
            r["rho_W"], yi, r["gene_name"], fontsize=S.FONT_ANNOT - 1, fontstyle="italic", zorder=10
        )
        for (_, r), yi in zip(top.iterrows(), y[top.index.values])
    ]
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    ax.text(
        0.02,
        0.96,
        f"{int((d.sig_perm == 'short').sum())}",
        transform=ax.transAxes,
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="left",
    )
    ax.text(
        0.98,
        0.96,
        f"{int((d.sig_perm == 'long').sum())}",
        transform=ax.transAxes,
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT - 0.5,
        fontweight="bold",
        va="top",
        ha="right",
    )
    ax.set_ylabel(r"$-\log_{10}p$ (nominal)", fontsize=S.FONT_TICK)
    ax.set_xlabel(r"Spearman $\rho_W$  (3$^\prime$UTR length vs pseudotime)", fontsize=S.FONT_TICK)
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    leg = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_SHORT,
            markersize=3.5,
            label="shortening",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_LONG,
            markersize=3.5,
            label="lengthening",
        ),
        Line2D(
            [0],
            [0],
            marker="*",
            color="w",
            markerfacecolor="none",
            markeredgecolor=ASD_STAR,
            markersize=5,
            label="ASD-risk",
        ),
    ]
    if compact:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1.5,
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(0.0, 0.86),
            handletextpad=0.1,
            labelspacing=0.15,
            borderpad=0.1,
        )
        fig.subplots_adjust(left=0.14, right=0.975, top=0.965, bottom=0.30)
    else:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1,
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=3,
            handletextpad=0.15,
            columnspacing=0.6,
        )
        ax.set_title(
            linlabel,
            fontsize=S.FONT_TICK,
            color=S.paper_palette_lineage[lin],
            fontweight="bold",
            pad=14,
        )
        fig.subplots_adjust(left=0.135, right=0.975, top=0.80, bottom=0.155)
    S.save_and_show(fig, save_name, figure="Fig2")


_volcano_lin(
    "excitatory",
    "Excitatory neurons",
    (S.HALF_W_IN, S.QUARTER_H_IN),
    "fig2c_pseudotime_volcano__pseudo_chisq_exc_quarter",
    compact=False,
)
_volcano_lin(
    "excitatory",
    "Excitatory neurons",
    (S.HALF_W_IN, S.EIGHTH_H_IN),
    "fig2c_pseudotime_volcano__pseudo_chisq_exc_eighth",
    compact=True,
)

#### Sensitivity: permutation-calibrated deflation (shaded = extrapolated past the perm floor)


In [ ]:
# Sensitivity: permutation-calibrated p (shaded band = extrapolated past the perm floor).
render_fig2c(
    "neglog_cal",
    r"$-\log_{10}p_{\mathrm{cal}}$  (perm-calibrated)",
    "fig2c_pseudotime_volcano__perm_calibrated",
    extrap=True,
)

### Variant: pseudo-chisq volcano, quarter-height half-width ('-log10 pseudo-p' y-label)


In [ ]:
# quarter-height x half-width version of the pseudo-chisq companion volcano; y-label matched to
# Fig1d ("-log10 pseudo-p"); compact 2-row layout (fewer gene labels, tighter spacing).
render_fig2c(
    "nl_chisq",
    r"$-\log_{10}$ pseudo-$p$",
    "fig2c_pseudotime_volcano__pseudo_chisq_quarter_half",
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
    ntop=2,
    compact=True,
)